# 08 微调（Fine-tuning） / Fine-tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiYuBY/use_token_to_learn/blob/main/docs/08-finetuning/08-finetuning.ipynb)

[📖 讲解 / Notes](https://baiyuby.github.io/use_token_to_learn/08-finetuning/README.html) · [🎮 交互演示 / Playground](https://baiyuby.github.io/use_token_to_learn/08-finetuning/interactive/finetune.html) · [🧑‍💻 GitHub](https://github.com/BaiYuBY/use_token_to_learn/tree/main/docs/08-finetuning)

这个 notebook 由 `scripts/make_notebooks.py` 根据本章 `code/` 目录下的脚本自动生成，代码与脚本完全一致：每个函数或类是一个单元格，
可以逐个修改、重新运行。最后一个单元格运行本章的演示（带 assert 检查）。

Generated by `scripts/make_notebooks.py` from this chapter's `code/` scripts, with identical code: one cell per function or
class, so you can edit and re-run them one at a time. The last cell runs the chapter's demo (with its assert checks).

In [ ]:
# 环境 / setup: find the repository, or clone it (e.g. in Colab); some chapters import code from earlier ones
# 找到仓库，找不到就克隆一份（例如在 Colab 中）；有些章节会导入前面章节的代码
import subprocess
import sys
from pathlib import Path

CHAPTER = "08-finetuning"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docs" / CHAPTER).is_dir()), None)
if ROOT is None:
    ROOT = Path.cwd() / "use_token_to_learn"
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BaiYuBY/use_token_to_learn.git", str(ROOT)], check=True)
CODE = ROOT / "docs" / CHAPTER / "code"
sys.path.insert(0, str(CODE))
print("code:", CODE)

## `lora.py`

```text
LoRA and QLoRA from scratch in PyTorch / 用 PyTorch 从零实现 LoRA 与 QLoRA.

Run / 运行:
    python docs/08-finetuning/code/lora.py

Covers / 包含:
    1. a LoRA layer: y = W x + (alpha / r) * B A x, with W frozen    LoRA 层：冻结 W，只训练 B、A
    2. merging B A into W, so inference costs nothing extra          把 B A 合并进 W，推理零额外开销
    3. the same layer via torch.nn.utils.parametrize, as a check     用 PyTorch 的 parametrize 再实现一遍做对比
    4. NF4, the 4-bit data type of QLoRA, vs plain 4-bit integers    QLoRA 的 4 位 NF4 格式与普通 INT4 对比
    5. how much GPU memory full fine-tuning, LoRA and QLoRA need     三种方法各需要多少显存
```

In [ ]:
__file__ = str(CODE / "lora.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.utils.parametrize as parametrize

### 1. LoRA layer / LoRA 层

In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear and learns a low-rank update  包住一个冻结的 nn.Linear，学习一个低秩更新.

        y = W x + b + (alpha / r) * B (A x),   A: r x d_in,  B: d_out x r

    A starts random and B starts at zero, so at step 0 the layer is exactly the original one.
    Initialization follows Hugging Face PEFT (Kaiming-uniform A, zero B).
    A 随机初始化、B 初始化为 0，所以训练开始时与原层完全相同。初始化方式与 Hugging Face PEFT 一致。
    """

    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        self.base.requires_grad_(False)
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T) @ self.B.T

    def delta_weight(self):
        return self.scale * self.B @ self.A

    def merge(self) -> nn.Linear:
        """Fold the update into an ordinary nn.Linear / 把更新合并成一个普通的 nn.Linear."""
        merged = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        with torch.no_grad():
            merged.weight.copy_(self.base.weight + self.delta_weight())
            if self.base.bias is not None:
                merged.bias.copy_(self.base.bias)
        return merged

In [ ]:
def add_lora(model, r, alpha, targets=("qkv", "out", "fc", "proj")):
    """Freeze the model and wrap every nn.Linear whose name is in targets.
    冻结整个模型，并把名字在 targets 中的 nn.Linear 换成 LoRA 层。"""
    model.requires_grad_(False)
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(module, name, LoRALinear(child, r, alpha))
    return model

In [ ]:
def merge_lora(model):
    """Replace every LoRA layer by its merged nn.Linear / 把每个 LoRA 层换回合并后的 nn.Linear."""
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if isinstance(child, LoRALinear):
                setattr(module, name, child.merge())
    return model

In [ ]:
def trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
def lora_params(d_in, d_out, r):
    """A has r * d_in entries and B has d_out * r / A 有 r*d_in 个参数，B 有 d_out*r 个."""
    return r * (d_in + d_out)

In [ ]:
class LoRAParametrization(nn.Module):
    def __init__(self, A, B, scale):
        super().__init__()
        self.A, self.B, self.scale = A, B, scale

    def forward(self, weight):
        return weight + self.scale * self.B @ self.A

In [ ]:
# The values used by bitsandbytes and the QLoRA paper / bitsandbytes 与 QLoRA 论文中的取值
NF4_TABLE = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
    -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
    0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0,
])

In [ ]:
def nf4_levels(offset=0.9677083):
    """Rebuild the NF4 table from normal quantiles / 用正态分位数重新算出 NF4 表.

    8 positive and 7 negative quantiles plus an exact zero, scaled into [-1, 1]. The asymmetry keeps
    an exact 0 while still using all 16 codes. `offset` is the outermost quantile used.
    8 个正分位点、7 个负分位点，再加上一个精确的 0，缩放到 [-1, 1]。不对称是为了既保留精确的 0，又用满 16 个码。
    """
    ndtri = torch.special.ndtri
    pos = ndtri(torch.linspace(offset, 0.5, 9, dtype=torch.float64)[:-1])
    neg = -ndtri(torch.linspace(offset, 0.5, 8, dtype=torch.float64)[:-1])
    levels = torch.cat([pos, torch.zeros(1, dtype=torch.float64), neg]).sort().values
    return (levels / levels.max()).float()

In [ ]:
INT4_TABLE = torch.linspace(-1, 1, 15)  # symmetric 4-bit integers -7..7, scaled / 对称的 4 位整数

In [ ]:
def quantize(weight, table, block=64):
    """Block-wise absmax quantization: each block of `block` numbers is divided by its largest
    magnitude, then each number is snapped to the nearest level in `table`.
    分块 absmax 量化：每 block 个数除以块内最大绝对值，再取 table 中最近的等级。"""
    flat = weight.reshape(-1, block)
    absmax = flat.abs().amax(dim=1, keepdim=True)
    codes = ((flat / absmax).unsqueeze(-1) - table).abs().argmin(dim=-1).to(torch.uint8)
    return codes, absmax

In [ ]:
def dequantize(codes, absmax, table, shape):
    return (table[codes.long()] * absmax).reshape(shape)

In [ ]:
class NF4Linear(nn.Module):
    """A frozen linear layer stored in NF4 and dequantized on the fly (the QLoRA base model).
    以 NF4 存储、计算时临时反量化的冻结线性层（QLoRA 的底座模型）。"""

    def __init__(self, base: nn.Linear, block=64):
        super().__init__()
        self.in_features, self.out_features, self.shape = base.in_features, base.out_features, base.weight.shape
        codes, absmax = quantize(base.weight.detach(), NF4_TABLE, block)
        self.register_buffer("codes", codes)
        self.register_buffer("absmax", absmax)
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)

    @property
    def weight(self):
        return dequantize(self.codes, self.absmax, NF4_TABLE, self.shape)

    def forward(self, x):
        return nn.functional.linear(x, self.weight, self.bias)

In [ ]:
class QLoRALinear(LoRALinear):
    """LoRA on top of an NF4 base layer / 底座为 NF4 的 LoRA 层."""

    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__(base, r, alpha)
        self.base = NF4Linear(base)

In [ ]:
def add_qlora(model, r, alpha, targets=("qkv", "out", "fc", "proj")):
    model.requires_grad_(False)
    for module in list(model.modules()):
        for name, child in list(module.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(module, name, QLoRALinear(child, r, alpha))
    return model

In [ ]:
def finetune_memory_gb(n_params, method, n_lora=0):
    """Bytes per parameter with mixed-precision AdamW / 混合精度 AdamW 下每个参数的字节数:

    full:  2 (bf16 weight) + 2 (bf16 grad) + 4 (fp32 master copy) + 8 (Adam m and v) = 16
    lora:  2 per frozen weight; the small LoRA weights pay the full 16
    qlora: 0.5 per frozen weight (4 bits) + about 0.127 bits of block constants after double
           quantization; the LoRA weights again pay 16
    """
    if method == "full":
        parts = {"weights": 2 * n_params, "gradients": 2 * n_params, "optimizer": 12 * n_params}
    elif method == "lora":
        parts = {"weights": 2 * n_params + 2 * n_lora, "gradients": 2 * n_lora, "optimizer": 12 * n_lora}
    elif method == "qlora":
        parts = {"weights": (4 + 0.127) / 8 * n_params + 2 * n_lora, "gradients": 2 * n_lora,
                 "optimizer": 12 * n_lora}
    else:
        raise ValueError(method)
    return {k: v / 1e9 for k, v in parts.items()}

In [ ]:
# Llama 2 7B: 32 layers, width 4096, FFN 11008, vocabulary 32000 / Llama 2 7B 的结构
LLAMA2_7B = dict(layers=32, d=4096, ffn=11008, vocab=32000)

In [ ]:
def llama_params(cfg):
    attn = 4 * cfg["d"] * cfg["d"]
    ffn = 3 * cfg["d"] * cfg["ffn"]  # SwiGLU: gate, up and down projections / 三个矩阵
    return cfg["layers"] * (attn + ffn) + 2 * cfg["vocab"] * cfg["d"]

In [ ]:
def llama_lora_params(cfg, r, targets):
    d, f = cfg["d"], cfg["ffn"]
    shapes = {"q": (d, d), "k": (d, d), "v": (d, d), "o": (d, d), "gate": (d, f), "up": (d, f), "down": (f, d)}
    return cfg["layers"] * sum(lora_params(*shapes[t], r) for t in targets)

In [ ]:
def main():
    torch.manual_seed(0)

    # 1. at initialization LoRA changes nothing (B = 0) / 初始化时 LoRA 不改变输出
    base = nn.Linear(64, 32)
    lora = LoRALinear(base, r=4, alpha=8)
    x = torch.randn(10, 64)
    assert torch.equal(lora(x), base(x))
    assert trainable_params(lora) == lora_params(64, 32, 4) == 384
    print(f"LoRA r=4 on a 64->32 layer: {trainable_params(lora)} trainable vs {base.weight.numel()} frozen  ✓")

    # 2. after some training the merged layer gives the same outputs / 训练后合并，输出不变
    nn.init.normal_(lora.B)  # pretend B has been trained / 假装 B 已经训练过
    merged = lora.merge()
    assert torch.allclose(merged(x), lora(x), atol=1e-5)
    assert torch.linalg.matrix_rank(lora.delta_weight()) == 4
    print("merged W + (alpha/r) B A matches the LoRA layer, and the update has rank 4  ✓")

    # 3. same thing with torch.nn.utils.parametrize / 用 parametrize 实现同样的层
    ref = nn.Linear(64, 32)
    ref.load_state_dict(base.state_dict())
    parametrize.register_parametrization(ref, "weight", LoRAParametrization(lora.A, lora.B, lora.scale))
    assert torch.allclose(ref(x), lora(x), atol=1e-5)
    print("LoRALinear matches a torch.nn.utils.parametrize implementation  ✓")

    # 4. NF4 / INT4 quantization error on normally distributed weights / 正态分布权重上的量化误差
    assert torch.allclose(nf4_levels(), NF4_TABLE, atol=1e-6)
    print("NF4 table rebuilt from normal quantiles matches bitsandbytes  ✓")
    w = torch.randn(1024, 1024) * 0.02
    err = {}
    for name, table in [("NF4", NF4_TABLE), ("INT4", INT4_TABLE)]:
        codes, absmax = quantize(w, table)
        err[name] = (dequantize(codes, absmax, table, w.shape) - w).pow(2).mean().sqrt() / w.std()
    assert err["NF4"] < err["INT4"]
    print(f"relative RMS error, block size 64: NF4 {err['NF4']:.3f}  INT4 {err['INT4']:.3f}  ✓ NF4 is lower")

    layer = nn.Linear(256, 256)
    q = NF4Linear(layer)
    rel = (q(x.repeat(1, 4)) - layer(x.repeat(1, 4))).norm() / layer(x.repeat(1, 4)).norm()
    print(f"NF4Linear output vs the original layer: {rel:.1%} relative difference")

    # 5. memory and parameter counts for a 7B model / 7B 模型的显存与参数量
    n = llama_params(LLAMA2_7B)
    assert abs(n - 6.74e9) < 0.01e9  # Llama 2 7B has 6.74B parameters / 官方参数量
    qv = llama_lora_params(LLAMA2_7B, 8, ["q", "v"])
    assert qv == 4_194_304  # 32 layers x 2 matrices x 8 x (4096 + 4096)
    every = llama_lora_params(LLAMA2_7B, 16, ["q", "k", "v", "o", "gate", "up", "down"])
    print(f"Llama 2 7B: {n / 1e9:.2f}B parameters; LoRA r=8 on q,v: {qv / 1e6:.1f}M ({qv / n:.3%}); "
          f"r=16 on every linear layer: {every / 1e6:.1f}M ({every / n:.2%})")
    for method in ["full", "lora", "qlora"]:
        mem = finetune_memory_gb(n, method, n_lora=every)
        print(f"  {method:>5}: {sum(mem.values()):6.1f} GB  " +
              "  ".join(f"{k} {v:.1f}" for k, v in mem.items()) + "  (+ activations)")

### 运行演示 / Run the demo (`lora.py`)

In [ ]:
main()

## `sft.py`

```text
Supervised fine-tuning (SFT) of a small pretrained model / 对一个预训练好的小模型做监督微调.

Run / 运行:
    python docs/08-finetuning/code/sft.py

Covers / 包含:
    1. pre-training a character-level GPT on Python source code      在 Python 源码上预训练字符级 GPT
    2. an instruction dataset in a fixed prompt template               固定模板的指令数据
    3. loss masking: learn only the response, not the prompt          损失掩码：只学回答，不学提问
    4. full fine-tuning vs LoRA vs QLoRA, and catastrophic forgetting  全参数微调、LoRA、QLoRA 与灾难性遗忘

The base model learns to continue Python code. After SFT it answers three kinds of instructions on
words it has never seen in training:
底座模型只会续写 Python 代码；SFT 之后，它要能对训练中没见过的单词完成三种指令：
    Upper: lambda   -> LAMBDA
    Reverse: lambda -> adbmal
    Length: lambda  -> 6
```

In [ ]:
__file__ = str(CODE / "sft.py")  # the code below finds its data files from here / 代码据此定位数据文件

In [ ]:
import copy
import math
import random
import re
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "07-scaling-laws" / "code"))

In [ ]:
from lora import add_lora, add_qlora, trainable_params  # noqa: E402
from scaling import load_corpus  # noqa: E402  (stdlib source, character level / 标准库源码，字符级)

In [ ]:
CTX = 128

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.norm1, self.norm2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)
        self.fc = nn.Linear(d, 4 * d)
        self.proj = nn.Linear(4 * d, d)

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = self.qkv(self.norm1(x)).view(b, t, 3, self.n_heads, d // self.n_heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.out(a.transpose(1, 2).reshape(b, t, d))
        return x + self.proj(F.gelu(self.fc(self.norm2(x))))

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d=96, n_layers=3, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(CTX, d)
        self.blocks = nn.ModuleList(Block(d, n_heads) for _ in range(n_layers))
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)
        self.head.weight = self.embed.weight  # weight tying / 权重共享
        for p in self.parameters():
            if p.dim() == 2:
                nn.init.normal_(p, std=0.02)

    def forward(self, ids):
        x = self.embed(ids) + self.pos(torch.arange(ids.size(1)))
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

### 1. Pre-training / 预训练

In [ ]:
class Corpus:
    def __init__(self, max_chars=4_000_000):
        self.train, self.val, self.vocab = load_corpus(max_chars)
        self.stoi = {c: i for i, c in enumerate(self.vocab)}
        g = torch.Generator().manual_seed(1)
        starts = torch.randint(len(self.val) - CTX - 1, (64,), generator=g)
        self.val_x = torch.stack([self.val[i:i + CTX] for i in starts])
        self.val_y = torch.stack([self.val[i + 1:i + CTX + 1] for i in starts])

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join(self.vocab[i] for i in ids)

    def text_batch(self, n, length, gen=None):
        starts = torch.randint(len(self.train) - length - 1, (n,), generator=gen)
        x = torch.stack([self.train[i:i + length] for i in starts])
        y = torch.stack([self.train[i + 1:i + length + 1] for i in starts])
        return x, y

In [ ]:
@torch.no_grad()
def code_loss(model, corpus):
    """Loss on held-out Python code: how much of the pre-training skill is left.
    在留出的 Python 代码上的损失：预训练学到的能力还剩多少。"""
    model.eval()
    loss = F.cross_entropy(model(corpus.val_x).flatten(0, 1), corpus.val_y.flatten()).item()
    model.train()
    return loss

In [ ]:
def pretrain(corpus, steps=2000, peak_lr=3e-3, batch=32, seed=0, log=print):
    torch.manual_seed(seed)
    model = GPT(len(corpus.vocab))
    opt = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)
    for step in range(steps):
        lr = peak_lr * min(1, (step + 1) / 100) * (0.1 + 0.45 * (1 + math.cos(math.pi * step / steps)))
        for group in opt.param_groups:
            group["lr"] = lr
        x, y = corpus.text_batch(batch, CTX)
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if log and (step % 500 == 0 or step == steps - 1):
            log(f"  pretrain step {step:>5}  loss {loss.item():.3f}  held-out code loss {code_loss(model, corpus):.3f}")
    return model

### 2. Instruction data / 指令数据

In [ ]:
TASKS = {
    "Upper": str.upper,
    "Reverse": lambda w: w[::-1],
    "Length": lambda w: str(len(w)),
}
TEMPLATE = "### Instruction:\n{task}: {word}\n### Response:\n"  # Alpaca-style / Alpaca 风格
END = "\n"  # the answer ends with a newline; generation stops there / 回答以换行结束
PAD_LEN = 64

In [ ]:
def split_words(corpus, seed=0):
    """Lower-case words from the code, split into train / validation / test with no overlap.
    从代码中取出小写单词，分成互不重叠的训练、验证、测试集。"""
    text = corpus.decode(corpus.train[:2_000_000].tolist())
    words = sorted(set(re.findall(r"\b[a-z]{3,8}\b", text)))
    random.Random(seed).shuffle(words)
    return {"test": words[:300], "val": words[300:600], "train": words[600:]}

In [ ]:
def make_example(corpus, task, word, mask_prompt=True):
    """Return (input ids, target ids). Prompt targets are -100 (ignored) when mask_prompt is on.
    返回输入与目标；开启掩码时，提问部分的目标设为 -100（不计入损失）。"""
    prompt = TEMPLATE.format(task=task, word=word)
    ids = corpus.encode(prompt + TASKS[task](word) + END)
    x, y = ids[:-1], ids[1:]
    if mask_prompt:
        y = [-100] * (len(prompt) - 1) + y[len(prompt) - 1:]
    return x + [0] * (PAD_LEN - len(x)), y + [-100] * (PAD_LEN - len(y))

In [ ]:
def sft_batch(corpus, words, rng, n, mask_prompt=True):
    pairs = [make_example(corpus, rng.choice(list(TASKS)), rng.choice(words), mask_prompt) for _ in range(n)]
    return torch.tensor([p[0] for p in pairs]), torch.tensor([p[1] for p in pairs])

In [ ]:
@torch.no_grad()
def generate(model, corpus, prompts, max_new=12):
    """Greedy decoding until the end marker; prompts of equal length are batched together.
    贪心解码直到结束符；等长的提示放在同一批。"""
    model.eval()
    outputs = {}
    by_len = {}
    for p in prompts:
        by_len.setdefault(len(p), []).append(p)
    for group in by_len.values():
        x = torch.tensor([corpus.encode(p) for p in group])
        texts, done = [""] * len(group), [False] * len(group)
        for _ in range(max_new):
            nxt = model(x[:, -CTX:])[:, -1].argmax(-1)
            x = torch.cat([x, nxt[:, None]], dim=1)
            for i, c in enumerate(corpus.decode(nxt.tolist())):
                if not done[i]:
                    done[i] = c == END
                    texts[i] += "" if done[i] else c
            if all(done):
                break
        outputs.update(zip(group, texts))
    model.train()
    return [outputs[p] for p in prompts]

In [ ]:
@torch.no_grad()
def response_loss(model, corpus, words):
    """Average loss on the response characters only / 只在回答部分计算的平均损失."""
    rng = random.Random(0)
    x, y = sft_batch(corpus, words, rng, 256)
    model.eval()
    loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten(), ignore_index=-100).item()
    model.train()
    return loss

In [ ]:
def accuracy(model, corpus, words):
    """Exact-match accuracy per task / 每个任务的完全匹配正确率."""
    result = {}
    for task, fn in TASKS.items():
        answers = generate(model, corpus, [TEMPLATE.format(task=task, word=w) for w in words])
        result[task] = sum(a == fn(w) for a, w in zip(answers, words)) / len(words)
    return result

### 3. Fine-tuning / 微调

In [ ]:
def prepare(base, method, r=8, alpha=None):
    """Copy the base model and set it up for one fine-tuning method / 复制底座并按方法设置.
    alpha defaults to 2r, a common choice / alpha 默认取 2r，是常见设置。"""
    model = copy.deepcopy(base)
    alpha = 2 * r if alpha is None else alpha
    if method == "lora":
        add_lora(model, r, alpha)
    elif method == "qlora":
        add_qlora(model, r, alpha)
    elif method != "full":
        raise ValueError(method)
    return model

In [ ]:
def finetune(model, corpus, words, steps=600, lr=1e-3, batch=32, mask_prompt=True, replay=0, seed=0,
             every=0, probe=None):
    """SFT with AdamW, short warmup, linear decay. `replay` extra rows of pre-training text are mixed
    into every batch to fight forgetting. Every `every` steps, probe(model) is recorded.
    AdamW + 短预热 + 线性衰减。replay 表示每批额外混入多少条预训练文本，用来对抗遗忘；
    每隔 every 步记录一次 probe(model)。"""
    torch.manual_seed(seed)
    rng = random.Random(seed)
    gen = torch.Generator().manual_seed(seed)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    history = []
    for step in range(steps):
        for group in opt.param_groups:
            group["lr"] = lr * min(1, (step + 1) / 30) * (1 - step / steps)
        x, y = sft_batch(corpus, words, rng, batch, mask_prompt)
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten(), ignore_index=-100)
        if replay:
            tx, ty = corpus.text_batch(replay, PAD_LEN, gen)
            loss = loss + F.cross_entropy(model(tx).flatten(0, 1), ty.flatten())
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        if every and (step + 1) % every == 0:
            history.append((step + 1, probe(model)))
    return history

In [ ]:
def main():
    torch.manual_seed(0)
    corpus = Corpus(max_chars=2_000_000)
    words = split_words(corpus)

    # the template and the loss mask / 模板与损失掩码
    x, y = make_example(corpus, "Reverse", "lambda")
    shown = "".join("·" if t == -100 else corpus.vocab[t] for t in y)
    print("example:", repr(corpus.decode(x).rstrip(corpus.vocab[0])))
    print("targets that count (· = masked):", repr(shown.rstrip("·")))
    assert corpus.decode([t for t in y if t != -100]) == "adbmal\n"

    print("pre-training a small GPT (quick version) / 预训练小模型（快速版）")
    base = pretrain(corpus, steps=300, log=print)
    before = response_loss(base, corpus, words["test"])
    print(f"base model: response loss {before:.2f}, accuracy {accuracy(base, corpus, words['test'][:100])}")

    for method in ["full", "lora", "qlora"]:
        model = prepare(base, method, r=8)
        finetune(model, corpus, words["train"], steps=250, lr=1e-3 if method == "full" else 1e-2)
        after = response_loss(model, corpus, words["test"])
        acc = {k: f"{v:.0%}" for k, v in accuracy(model, corpus, words["test"][:100]).items()}
        print(f"{method:>5}: {trainable_params(model):>7,} trainable params  response loss {after:.2f}  "
              f"accuracy {acc}  code loss {code_loss(base, corpus):.2f} -> {code_loss(model, corpus):.2f}")
        assert after < 0.7 * before  # it learned to answer (response loss drops 4.4 -> about 2) / 学会了回答

    print("sample / 示例:", generate(model, corpus, [TEMPLATE.format(task="Length", word=w) for w in ["token", "learn"]]))

### 运行演示 / Run the demo (`sft.py`)

In [ ]:
main()

## 动手试试 / Try it yourself

改一改上面任意一个函数或参数，重新运行它所在的单元格，再运行 `main()` 看结果有什么变化。
assert 失败说明你的改动改变了程序检查的性质，这正是值得思考的地方。

Change any function or parameter above, re-run its cell, then run `main()` again. If an assert fails, your change broke a
property the script checks, which is exactly what is worth thinking about.